# Deploying Granite 4.2 30B in Amazon SageMaker

## Introduction to Granite 4.2 30B

IBM Granite 4.2 30B is a dense, instruction-tuned language model with about 30 billion parameters, released under the Apache 2.0 license. It is optimized for:
- Instruction following and chat
- Reasoning (the model can "think" before it answers)
- Tool use / function calling
- Summarization, classification, entity extraction, and RAG
- Long-context input (128K tokens native context)

The model is available on AWS Marketplace and on [Hugging Face](https://huggingface.co/ibm-granite/granite-4.2-30b).

---

## Pre-requisites

- Use this notebook from an **Amazon SageMaker Notebook Instance** or **SageMaker Studio**.
- Ensure your execution IAM role has **AmazonSageMakerFullAccess**.
- Subscribe to the model package on AWS Marketplace.
- Make sure your account has quota for **ml.g7.24xlarge** endpoint usage in your region.

---

## Contents
1. Subscribe to and select the Granite 4.2 30B model package
2. Create an endpoint
3. Run real-time inference (thinking on, thinking off, tool calling, long context)
4. Clean up resources

---

### 1. Subscribe to the model package

1. Go to [IBM Granite 4.2 30B on AWS Marketplace](https://aws.amazon.com/marketplace).
2. Click **Continue to subscribe**.
3. Accept the offer, choose your region, and copy the **Model Package ARN**.
4. Paste the ARN into the `model_package_map` below for your region.

### 1. Select the model package

Confirm that you received this notebook from the model catalog on the SageMaker AWS Management Console. This model package is published in **us-east-1**.

In [ ]:
model_package_map = {
    "us-east-1": "<MODEL_PACKAGE_ARN>",   # paste the Model Package ARN from your AWS Marketplace subscription
}

In [ ]:
# SageMaker Python SDK v2 is required: v3 moved ModelPackage and get_execution_role.
# After this cell finishes, restart the kernel once, then run from the next cell.
!pip install --upgrade pip
!pip install -q --upgrade boto3 "sagemaker>=2.240,<3"

In [ ]:
import json
import pprint
from datetime import datetime

import boto3
import sagemaker
from sagemaker import ModelPackage, get_execution_role

In [ ]:
sagemaker_session = sagemaker.Session()

try:
    execution_role_arn = sagemaker.get_execution_role()
except ValueError:
    execution_role_arn = None

if execution_role_arn == None:
    execution_role_arn = input("Enter your execution role ARN: ")

region = sagemaker_session.boto_region_name
runtime_sm_client = boto3.client("sagemaker-runtime")

print ("execution_role_arn: ", execution_role_arn)
print ("region: ", region)

In [ ]:
if region not in model_package_map.keys():
    raise ValueError("UNSUPPORTED REGION")

model_package_arn = model_package_map[region]
if model_package_arn.startswith("<"):
    raise ValueError("Set your Model Package ARN in model_package_map above")

print ("model_package_arn: ", model_package_arn)

## Create an endpoint and perform real-time inference

In this example, we deploy IBM Granite 4.2 30B on an Amazon SageMaker real-time endpoint hosted on a GPU instance.
If you need general information on real-time inference with Amazon SageMaker, please refer to the SageMaker documentation.

Granite 4.2 30B is a dense transformer in BF16 (about 59 GB of weights). The model package is pre-configured to run with tensor parallelism across 4 GPUs and a 128K token context window, so it needs a multi-GPU instance.
In this example we use an **ml.g7.24xlarge** instance (4 GPUs, 128 GB total GPU memory). This is the instance type this model package has been tested on.

The endpoint uses the vLLM inference server and accepts OpenAI-style chat requests (a JSON body with `messages`) on the SageMaker `/invocations` route.

The IBM Granite 4.2 30B model supports:
- Instruction-following tasks
- Optional step-by-step reasoning (returned separately in the `reasoning` field)
- Tool / function calling
- Long-context input handling (up to 128K tokens)

### 2. Define the endpoint configuration

In [ ]:
model_name = "granite-4-2-30b"
inference_instance_type = "ml.g7.24xlarge"
# The model package needs this host image version (CUDA 13 / newer GPU driver).
inference_ami_version = "al2023-ami-sagemaker-inference-gpu-4-1"
model_download_timeout = 3600
health_check_timeout = 1800
instance_count = 1

### 3. Create the endpoint

In [ ]:
# create a deployable model from the model package.
model = ModelPackage(
    role=execution_role_arn, model_package_arn=model_package_arn, sagemaker_session=sagemaker_session
)

# create a unique endpoint name
timestamp = "{:%Y-%m-%d-%H-%M-%S}".format(datetime.now())
endpoint_name = f"{model_name}-{timestamp}"
print(f"Deploying endpoint {endpoint_name}")

In [ ]:
# deploy the model
deployed_model = model.deploy(
    initial_instance_count=instance_count,
    instance_type=inference_instance_type,
    endpoint_name=endpoint_name,
    inference_ami_version=inference_ami_version,
    model_data_download_timeout=model_download_timeout,
    container_startup_health_check_timeout=health_check_timeout,
)

If you have already deployed your model, you can also access it via your chosen endpoint_name and sagemaker_session:

In [ ]:
deployed_model = sagemaker.Predictor(
    endpoint_name=endpoint_name,
    sagemaker_session=sagemaker_session,
)

SageMaker will now create our endpoint and deploy the model to it. Downloading about 59 GB of weights and loading them onto 4 GPUs can take 15-30 minutes. Once the endpoint is in service, you will be able to perform real-time inference.

## Run inference with the model

Now that we have the Granite 4.2 30B model deployed to a SageMaker endpoint, we can start testing. We send OpenAI-style chat requests with `invoke_endpoint`. Generation parameters such as `temperature`, `top_p` and `max_tokens` are top-level fields of the request body. IBM recommends `temperature` 1.0 and `top_p` 0.95 for this model.

In [ ]:
def invoke(body):
    r = runtime_sm_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(body),
    )
    return json.loads(r["Body"].read())

question = "A farmer has 10 cows, 5 chickens, and 2 horses. How many legs are there in total?"

### 4. Example request: chat with thinking on (default)

By default the model thinks before it answers. The thinking text is returned in the message `reasoning` field and the final answer in `content`.

In [ ]:
out = invoke({
    "messages": [{"role": "user", "content": question}],
    "temperature": 1.0, "top_p": 0.95, "max_tokens": 4096,
})
msg = out["choices"][0]["message"]
print("reasoning:", (msg.get("reasoning") or msg.get("reasoning_content") or "")[:500])
print("content:", msg["content"])
print(out["usage"])

### 5. Example request: thinking off

Pass `chat_template_kwargs` with `enable_thinking` set to `False` to get a direct answer without the thinking phase.

In [ ]:
out = invoke({
    "messages": [{"role": "user", "content": question}],
    "temperature": 1.0, "top_p": 0.95, "max_tokens": 1024,
    "chat_template_kwargs": {"enable_thinking": False},
})
print(out["choices"][0]["message"]["content"])

### 6. Example request: tool calling

In [ ]:
tools = [{"type": "function", "function": {
    "name": "get_current_weather", "description": "Get the current weather in a city",
    "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}}]
out = invoke({
    "messages": [{"role": "user", "content": "What is the weather in Boston right now?"}],
    "tools": tools, "tool_choice": "auto",
    "temperature": 1.0, "top_p": 0.95, "max_tokens": 2048,
})
print(json.dumps(out["choices"][0]["message"], indent=2))

### 7. Example request: long-context retrieval

A secret code is hidden in the middle of about 100K tokens of filler text, and the model must retrieve it. This checks that the 128K context window is working.

In [ ]:
# One filler sentence is about 10 tokens. Check out["usage"]["prompt_tokens"] for the real size.
TARGET_TOKENS = 100_000
needle = "The secret access code is MANGO-7421."
sentence = "The quick brown fox jumps over the lazy dog. "
n = TARGET_TOKENS // 10
filler = sentence * (n // 2) + needle + " " + sentence * (n // 2)
out = invoke({
    "messages": [{"role": "user", "content": filler + "\n\nWhat is the secret access code? Answer with the code only."}],
    "temperature": 0.0, "max_tokens": 64,
    "chat_template_kwargs": {"enable_thinking": False},
})
answer = out["choices"][0]["message"]["content"]
print(answer, out["usage"])
assert "MANGO-7421" in answer, "needle not retrieved"

## Clean-up

Please don't forget to run the cells below to delete all resources and avoid unnecessary charges.

### 8. Delete the endpoint

In [ ]:
model.sagemaker_session.delete_endpoint(endpoint_name)
model.sagemaker_session.delete_endpoint_config(endpoint_name)

### 9. Delete the model

In [ ]:
model.delete_model()

Thank you for trying out IBM Granite 4.2 30B on SageMaker. We have only scratched the surface of what you can do with this model.

Welcome to your IBM Granite Model support experience! You can view, start, or contribute to community discussions (sign in to contribute). View supplemental resources and  [sign](https://www.ibm.com/mysupport/s/?language=en_US) in to open a new case.

## Would you like to provide feedback?

Please let us know your comments about our family of Granite models by visiting our collection. Select the repository of the model you would like to provide feedback about. Then, go to Community tab, and click on New discussion. Alternatively, you can also post any questions/comments on our github discussions page.